# Deep Semantic Search — Full Demo

This notebook demonstrates all the functionality of the `deep-semantic-search` library:

1. **Image Search** — Search images by text or by image similarity
2. **Text Search** — Semantic text search over documents
3. **Image Clustering** — Cluster images using KMeans on CLIP embeddings
4. **Image Captioning** — Generate captions for images using BLIP
5. **RAG (Question Answering)** — Ask questions over text documents

## Setup

Make sure the package is installed:
```bash
pip install deep-semantic-search
# or from source:
pip install -e ..
```

In [ ]:
import os
from pathlib import Path

# Paths to demo data
DEMO_DIR = Path(".").resolve()
IMAGE_DIR = DEMO_DIR / "data" / "images"
TEXT_DIR = DEMO_DIR / "data" / "texts"

print(f"Demo directory: {DEMO_DIR}")
print(f"Image directory: {IMAGE_DIR} ({len(list(IMAGE_DIR.iterdir()))} files)")
print(f"Text directory: {TEXT_DIR} ({len(list(TEXT_DIR.iterdir()))} files)")

## 1. Image Search

Load images, build a CLIP index, and search by text or by image.

In [ ]:
from deep_semantic_search import LoadImageData, ImageIndexer, ImageSearcher

# Load all images from the demo folder
loader = LoadImageData()
image_paths = loader.from_folder([str(IMAGE_DIR)])
print(f"Loaded {len(image_paths)} images")
for p in image_paths:
    print(f"  {os.path.basename(p)}")

In [ ]:
# Build the CLIP index (auto-skips if already built)
indexer = ImageIndexer(
    image_paths,
    metadata_dir=DEMO_DIR / "metadata" / "clip_index"
)
indexer.run_index(reindex=True)
print(f"Indexed {len(indexer.image_data)} images")

In [ ]:
# Search by text query
searcher = ImageSearcher(indexer)
results = searcher.search_by_text("a blue sky with water", n=5)

print("\nSearch results for 'a blue sky with water':")
for r in results:
    print(f"  {r['score']:.4f}  {os.path.basename(r['path'])}")

In [ ]:
# Search by text — different query
results = searcher.search_by_text("warm red colors, sunset", n=5)

print("\nSearch results for 'warm red colors, sunset':")
for r in results:
    print(f"  {r['score']:.4f}  {os.path.basename(r['path'])}")

In [ ]:
# Search by image — find images similar to the first one
query_image = image_paths[0]
results = searcher.search_by_image(query_image, n=5)

print(f"\nImages similar to '{os.path.basename(query_image)}':")
for r in results:
    print(f"  {r['score']:.4f}  {os.path.basename(r['path'])}")

In [ ]:
# Visualize similar images (opens matplotlib plots)
searcher.plot_similar_images(query_image, n=4)

## 2. Text Search

Load text documents, create embeddings, and search by semantic similarity.

In [ ]:
from deep_semantic_search import LoadTextData, TextEmbedder, TextSearch

# Load text files
text_loader = LoadTextData()
corpus = text_loader.from_folder(str(TEXT_DIR))

print(f"Loaded {len(corpus)} documents:")
for path, text in corpus.items():
    preview = text[:80].replace('\n', ' ')
    print(f"  {os.path.basename(path)}: \"{preview}...\"")

In [ ]:
# Create sentence embeddings
embedder = TextEmbedder(
    metadata_dir=DEMO_DIR / "metadata" / "text_embeddings"
)
embedder.embed(corpus, reindex=True)
print("Embeddings created successfully!")

In [ ]:
# Search for similar documents
search = TextSearch(embedder)

queries = [
    "How does CLIP work for image understanding?",
    "What is retrieval augmented generation?",
    "Which Python libraries are popular for AI?",
    "How do neural networks learn from data?",
]

for query in queries:
    results = search.find_similar(query, top_n=3)
    print(f"\nQuery: '{query}'")
    for r in results:
        print(f"  Score: {r['score']:.4f}  File: {os.path.basename(r['path'])}")
        print(f"         \"{r['text'][:100].replace(chr(10), ' ')}...\"")

## 3. Image Clustering

Cluster images into groups using KMeans on CLIP feature vectors.

In [ ]:
from deep_semantic_search import ImageClusterer

# Cluster images into 3 groups
clusterer = ImageClusterer(indexer)
cluster_df = clusterer.cluster(n_clusters=3)

print("Clustering results:")
print(cluster_df[["images_paths", "cluster", "topic"]].to_string(index=False))

In [ ]:
# View images per cluster
for cluster_id in sorted(cluster_df["cluster"].unique()):
    images = clusterer.get_cluster_images(int(cluster_id))
    print(f"\nCluster {cluster_id} ({len(images)} images):")
    for img in images:
        print(f"  {os.path.basename(img)}")

In [ ]:
# Save clusters to organized folders
save_dir = DEMO_DIR / "output" / "clusters"
clusterer.save_clusters(str(save_dir))
print(f"Clusters saved to {save_dir}")

for d in sorted(save_dir.iterdir()):
    files = list(d.iterdir())
    print(f"  {d.name}/  ({len(files)} images)")

In [ ]:
# Plot a cluster
clusterer.plot_cluster(0, n=6)

## 4. Image Captioning

Generate natural language captions for images using BLIP.

> **Note:** This downloads the BLIP model (~1.5 GB) on first run.

In [ ]:
from deep_semantic_search import ImageCaptioner

captioner = ImageCaptioner()

# Caption a subset of images
sample_paths = image_paths[:5]
captions_df = captioner.caption(sample_paths)

print("Image Captions:")
for _, row in captions_df.iterrows():
    print(f"  {os.path.basename(row['image_path'])}: {row['caption']}")

In [ ]:
# Visualize captioned images
captioner.plot_captioned_images(captions_df, caption_col="caption")

## 4b. Clustering with Captioning

Combine clustering with BLIP captioning to auto-generate topic labels.

In [ ]:
# Cluster with captioner for automatic topic labels
# Note: This requires Ollama running locally for LLM-based topic extraction
# If Ollama is not available, topics will default to generic labels
clusterer_with_topics = ImageClusterer(indexer)
cluster_df_captioned = clusterer_with_topics.cluster(n_clusters=3, captioner=captioner)

print("Clusters with auto-generated topics:")
for cluster_id in sorted(cluster_df_captioned["cluster"].unique()):
    cluster_data = cluster_df_captioned[cluster_df_captioned["cluster"] == cluster_id]
    topic = cluster_data["topic"].iloc[0]
    count = len(cluster_data)
    print(f"  Cluster {cluster_id} — Topic: '{topic}' ({count} images)")

## 5. RAG — Question Answering

Answer questions over text documents using Retrieval-Augmented Generation.

> **Note:** Requires Ollama running locally with a model (default: `gemma4:e4b`).
> 
> ```bash
> # Install Ollama: https://ollama.ai
> ollama pull gemma4:e4b
> ```

In [ ]:
from deep_semantic_search import ask_question

# Get all text content
text_data = list(corpus.values())
print(f"Using {len(text_data)} documents for RAG")

# Ask a question
question = "What is CLIP and how is it used for image search?"
print(f"\nQuestion: {question}")

try:
    answer = ask_question(text_data, question)
    print(f"\nAnswer: {answer}")
except Exception as e:
    print(f"\nRAG requires Ollama to be running. Error: {e}")

In [ ]:
# Ask another question
question2 = "What are the steps in a RAG pipeline?"
print(f"Question: {question2}")

try:
    answer2 = ask_question(text_data, question2)
    print(f"\nAnswer: {answer2}")
except Exception as e:
    print(f"\nRAG requires Ollama to be running. Error: {e}")

In [ ]:
# RAG with custom parameters
question3 = "What Python libraries are useful for machine learning?"
print(f"Question: {question3}")

try:
    answer3 = ask_question(
        text_data,
        question3,
        chunk_size=500,
        chunk_overlap=50,
    )
    print(f"\nAnswer: {answer3}")
except Exception as e:
    print(f"\nRAG requires Ollama to be running. Error: {e}")

## 6. CLI Usage

The package also includes a `dss` CLI tool. Here are equivalent shell commands:

```bash
# Image search
dss image-search --folder ./data/images --query "blue sky" --top 5

# Text search
dss text-search --folder ./data/texts "neural networks" --top 3

# Image clustering
dss image-cluster --folder ./data/images --clusters 3 --save-dir ./output/clusters

# RAG question answering
dss ask --folder ./data/texts "What is CLIP?"
```

## Cleanup

Run this cell to remove generated metadata and output files.

In [ ]:
import shutil

for d in ["metadata", "output"]:
    path = DEMO_DIR / d
    if path.exists():
        shutil.rmtree(path)
        print(f"Removed {path}")

print("Cleanup complete!")